## 퍼널 분석 (Funnel Analysis)
- 목적: 주문~리뷰까지 각 단계별 소요시간 측정, 병목 구간 탐지
- 핵심 질문: 판매자가 택배사에 늦게 준 것인가, 택배사가 배송을 오래 한 것인가?
- 사용 컬럼: order_purchase_timestamp · order_approved_at · order_delivered_carrier_date · order_delivered_customer_date
- 시각화: 구간별 소요시간 히스토그램 · 박스플롯

## 코호트 분석 (Cohort Analysis)
- 목적: 배송 지연 경험 고객의 재구매율 추적
- 가설: 배송 지연 경험 고객은 정상 배송 고객보다 리텐션이 낮을 것이다
- 사용 컬럼: customer_unique_id · purchase_month · is_late · review_score
- 시각화: 재구매율 히트맵 · 그룹별 리텐션 곡선

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATE_COLS = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

df = pd.read_csv("funnel_df.csv", parse_dates=DATE_COLS)
valid = df[df['is_valid_funnel']].copy()

print('전체:', df.shape)
print('유효 주문:', valid.shape)
print()
print(valid[['t_approve_d','t_carrier_d','t_delivery_d']].describe().round(2))

전체: (99441, 18)
유효 주문: (95088, 18)

       t_approve_d  t_carrier_d  t_delivery_d
count     95088.00     95088.00      95088.00
mean          0.40         2.85          9.36
std           0.80         3.48          8.77
min           0.00         0.00          0.00
25%           0.01         0.90          4.11
50%           0.01         1.85          7.11
75%           0.56         3.62         12.06
max          30.89       125.76        205.19


In [6]:
# 1. order_status 확인
print('is_valid_funnel vs delivered 비교')
print('is_valid_funnel True:', df['is_valid_funnel'].sum())
print('delivered 건수:', (df['order_status'] == 'delivered').sum())
print()

# 2. valid 기준으로 확정
valid = df[df['is_valid_funnel']].copy()

# 3. review_score 결측 확인
print('review_score 결측:', valid['review_score'].isnull().sum())

# 4. outlier 확인
print()
print('t_carrier_d 99퍼센타일:', valid['t_carrier_d'].quantile(0.99).round(2))
print('t_delivery_d 99퍼센타일:', valid['t_delivery_d'].quantile(0.99).round(2))

is_valid_funnel vs delivered 비교
is_valid_funnel True: 95088
delivered 건수: 96478

review_score 결측: 639

t_carrier_d 99퍼센타일: 17.14
t_delivery_d 99퍼센타일: 41.03
